In [1]:
import os
from dotenv import load_dotenv
import json
import gradio as gr
import google.generativeai as genai

In [2]:
load_dotenv(override = True)
api_key = os.getenv("GEM_API_KEY")
genai.configure(api_key=api_key)

In [3]:
model = genai.GenerativeModel("gemini-2.5-flash")

In [4]:
system_message = "You are a very helpful Airline Assistant called FlightAI"

In [8]:
def chat(message,history):
    messages = [{"role":"system", "content":system_message}]
    for human, assistant in history:
        messages.append({"role":"user", "content":human})
        messages.append({"role":"assistant","content":assistant})
    messages.append({"role":"user","content":message})
    response = model.generate_content(system_message)
    return response.text

gr.ChatInterface(fn=chat).launch()

C:\Users\hp\anaconda3\Lib\site-packages\gradio\chat_interface.py:345: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


In [13]:
ticket_price = {"london":"$799", "paris":"$899", "tokyo":"$1400","usa":"$999"}

def get_ticket(destination_city):
    city = destination_city.lower()
    return ticket_price.get(city,"unknown")

In [15]:
get_ticket("london")

'$799'

In [16]:
"""
Dictionary specification describing the `get_ticket_price` function for use in function calling APIs.

This structure follows the JSON Schema format and is typically used with AI assistants 
(e.g., OpenAI, Gemini) to allow them to programmatically call a predefined function 
when a specific intent is detected, such as asking for ticket prices.

Attributes:
    name (str): The function's identifier ("get_ticket_price").
    description (str): Explains the purpose of the function and when it should be used.
    parameters (dict): JSON Schema defining the expected input parameters:
        type (str): Always "object" to indicate the function takes a JSON object as input.
        properties (dict):
            destination_city (dict):
                type (str): The city the user wants to travel to.
                description (str): Explains that this is the destination for ticket pricing.
        required (list): Specifies that "destination_city" must be provided.
        additionalProperties (bool): Disallows parameters other than those defined.
"""
price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city. Call this whenever you need to know the ticket price, for example when a customer asks 'How much is a ticket to this city'",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

In [17]:
tools = [{"type": "function", "function": price_function}]

In [18]:
def chat(message, history):
    """
    Handles a single turn of chat with the Gemini API, including tool call handling.

    Args:
        message (str): The latest user message.
        history (list): Conversation history in Gemini-compatible role/parts format.

    Returns:
        str: Model's final reply text.
    """

    messages = [{"role": "user", "parts": [system_message]}]

    
    for msg in history:
        if msg["role"] == "user":
            messages.append({"role": "user", "parts": [msg["content"]]})
        elif msg["role"] == "assistant":
            messages.append({"role": "model", "parts": [msg["content"]]})


    messages.append({"role": "user", "parts": [message]})
    response = model.generate_content(messages)

    if response.candidates and "functionCall" in str(response):
        # Example: Detecting a function call from the response
        tool_response, city = handle_tool_call(response)
        messages.append({"role": "model", "parts": [str(response.text)]})
        messages.append({"role": "user", "parts": [tool_response]})
        response = model.generate_content(messages)

    return (response.text or "").strip()


In [19]:
def handle_tool_call(response):
    """
    Handles a Gemini function/tool call by parsing the model's structured output
    and returning the tool's execution result.

    Args:
        response: Gemini model's response object from `generate_content`.

    Returns:
        tuple: (tool_response_dict, city)
            tool_response_dict -> dict containing the tool's reply for the conversation
            city -> str, extracted destination city
    """
    parts = response.candidates[0].content.parts

    function_call = None
    for part in parts:
        if "functionCall" in part:
            function_call = part["functionCall"]
            break

    if not function_call:
        raise ValueError("No function call found in Gemini response")

    args = function_call.get("args", {})
    city = args.get("destination_city")

    price = get_ticket_price(city)

    tool_response = {
        "role": "function",
        "parts": [{
            "functionResponse": {
                "name": function_call["name"],
                "response": {"destination_city": city, "price": price}
            }
        }]
    }
    return tool_response, city


In [20]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.
